In [0]:
%sql
--------------------------------------------------------------------------------------------------------------
--- Adapted to Databricks SQL drug_era from Pure SQL drug_era written by Chris_Knoll:
--- https://gist.github.com/chrisknoll/c820cc12d833db2e3d1e
--- Upgraded to v5
--- Uses STOCKPILE method to populate gap_days field
--- INTERVAL set to 30 days
---
--- Chris Knoll's comments are after two dashes
--- Taylor Delehanty's comments are after three dashes
--- proper schema for "<schema>" needs to be replaced in the code
--- proper schema for "<vocabulary_and_concept_schema>" needs to be replaced in the code.
--- This schema is where the vocabularies and concepts are located.
--------------------------------------------------------------------------------------------------------------

TRUNCATE TABLE _exponent.omop_tw.drug_era;



In [0]:
%sql
WITH cteDrugPreTarget AS
(
  -- Normalize DRUG_EXPOSURE_END_DATE to either the existing drug exposure end date,
  -- or add days supply, or add 1 day to the start date
  SELECT
      d.drug_exposure_id,
      d.person_id,
      c.concept_id AS ingredient_concept_id,
      d.drug_exposure_start_date AS drug_exposure_start_date,
      d.days_supply AS days_supply,

      COALESCE(
        --- If drug_exposure_end_date is present, use it
        d.drug_exposure_end_date,

        --- Else if days_supply is present and > 0, use start_date + days_supply
        CASE
          WHEN d.days_supply IS NOT NULL AND d.days_supply > 0
            THEN date_add(d.drug_exposure_start_date, CAST(d.days_supply AS INT))
          ELSE NULL
        END,

        --- Else add 1 day to the drug_exposure_start_date since there is no end_date or usable days_supply
        date_add(d.drug_exposure_start_date, 1)
      ) AS drug_exposure_end_date

  FROM _exponent.omop_tw.drug_exposure d
    -- JOIN _exponent.omop.concept_ancestor ca
    --   ON ca.descendant_concept_id = d.drug_concept_id
    JOIN _exponent.omop.concept c
      ON d.drug_concept_id = c.concept_id

  --- NOTE: In OMOP, vocabulary_id is typically a STRING (e.g., 'RxNorm')
  WHERE c.vocabulary_id = 'RxNorm'
  AND c.concept_class_id = 'Ingredient'

  /* Depending on the needs of your data, you can put more filters on to your code.
   * We assign 0 to unmapped drug_concept_id's, and we found data where days_supply was negative.
   * We don't want different drugs put in the same era, so the code below shows how we filtered them out.
   * We also don't want negative days_supply, because that will pull our end_date before the start_date.
   * For now, we are filtering those out as well, but this is a data quality issue that we are trying to solve.
   */
  --- AND d.drug_concept_id != 0
  --- AND (d.days_supply IS NULL OR d.days_supply >= 0)
)

--------------------------------------------------------------------------------------------------------------
, cteDrugTarget AS
(
  SELECT
      drug_exposure_id,
      person_id,
      ingredient_concept_id,
      drug_exposure_start_date,
      days_supply,
      drug_exposure_end_date,

      --- Calculates the days of exposure to the drug so at the end we can subtract the SUM of these days
      --- from the total days in the era.
      datediff(drug_exposure_end_date, drug_exposure_start_date) AS days_of_exposure
  FROM cteDrugPreTarget
)

--------------------------------------------------------------------------------------------------------------
, cteEndDates AS -- the magic
(
  SELECT
      person_id,
      ingredient_concept_id,
      date_add(event_date, -30) AS end_date  -- unpad the end date
  FROM
  (
    SELECT
        person_id,
        ingredient_concept_id,
        event_date,
        event_type,

        -- this pulls the current START down from the prior rows so that the NULLs from the END DATES
        -- will contain a value we can compare with
        MAX(start_ordinal) OVER (
          PARTITION BY person_id, ingredient_concept_id
          ORDER BY event_date, event_type
          ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS start_ordinal,

        -- this re-numbers the inner UNION so all rows are numbered ordered by the event date
        ROW_NUMBER() OVER (
          PARTITION BY person_id, ingredient_concept_id
          ORDER BY event_date, event_type
        ) AS overall_ord
    FROM
    (
      -- select the start dates, assigning a row number to each
      SELECT
          person_id,
          ingredient_concept_id,
          drug_exposure_start_date AS event_date,
          -1 AS event_type,
          ROW_NUMBER() OVER (
            PARTITION BY person_id, ingredient_concept_id
            ORDER BY drug_exposure_start_date
          ) AS start_ordinal
      FROM cteDrugTarget

      UNION ALL

      -- pad the end dates by 30 to allow a grace period for overlapping ranges.
      SELECT
          person_id,
          ingredient_concept_id,
          date_add(drug_exposure_end_date, 30) AS event_date,
          1 AS event_type,
          NULL AS start_ordinal
      FROM cteDrugTarget
    ) RAWDATA
  ) e
  WHERE (2 * e.start_ordinal) - e.overall_ord = 0
)

--------------------------------------------------------------------------------------------------------------
, cteDrugExposureEnds AS
(
  SELECT
      dt.person_id,
      dt.ingredient_concept_id AS drug_concept_id,
      dt.drug_exposure_start_date,
      MIN(e.end_date) AS drug_era_end_date,
      dt.days_of_exposure AS days_of_exposure
  FROM cteDrugTarget dt
  JOIN cteEndDates e
    ON dt.person_id = e.person_id
   AND dt.ingredient_concept_id = e.ingredient_concept_id
   AND e.end_date >= dt.drug_exposure_start_date
  GROUP BY
      dt.drug_exposure_id,
      dt.person_id,
      dt.ingredient_concept_id,
      dt.drug_exposure_start_date,
      dt.days_of_exposure
)

--------------------------------------------------------------------------------------------------------------
INSERT INTO _exponent.omop_tw.drug_era (
  person_id,
  drug_concept_id,
  drug_era_start_date,
  drug_era_end_date,
  drug_exposure_count,
  gap_days
)
SELECT
    person_id,
    drug_concept_id,
    MIN(drug_exposure_start_date) AS drug_era_start_date,
    drug_era_end_date,
    COUNT(*) AS drug_exposure_count,

    --- STOCKPILE gap_days:
    --- total era days - sum(exposure days)
    --- dividing by 86400 in Postgres converts seconds -> days; in Spark we can compute days directly
    datediff(drug_era_end_date, MIN(drug_exposure_start_date)) - SUM(days_of_exposure) AS gap_days

FROM cteDrugExposureEnds
GROUP BY person_id, drug_concept_id, drug_era_end_date
ORDER BY person_id, drug_concept_id
;

 /*
 --- This is a common test to make sure you have the same number of exposures going in as contribute to the count at the end.
 --- Make sure the JOIN and AND statements are the same as above so that your counts actually represent what you should be getting.
 SELECT
   (SELECT COUNT(*)
    FROM _exponent.omop.drug_exposure d
    JOIN _exponent.omop.concept_ancestor ca ON ca.descendant_concept_id = d.drug_concept_id
    JOIN _exponent.omop.concept c ON ca.ancestor_concept_id = c.concept_id
    WHERE c.vocabulary_id = 'RxNorm'
      AND c.concept_class_id = 'Ingredient'
      AND d.drug_concept_id != 0 --- Our unmapped drug_concept_id's are set to 0, so we don't want different drugs wrapped up in the same era
      AND d.days_supply >= 0) AS count,
   (SELECT SUM(drug_exposure_count) FROM _exponent.omop.drug_era) AS sum
 */